# LIBERO — **10 task 각각 성공 영상 1개**

학습된 ours(`bimamba_s7`) 체크포인트로, LIBERO-10의 **10개 task마다 성공한 에피소드 영상 1개**를 뽑는다.

- `RECORD_VIDEOS=1` + `RECORD_SUCCESS_ONLY=1` → **성공한 에피소드만** 저장(실패는 건너뜀). 파일명 `..._success.mp4`.
- 성공 1개 나오면 그 task 렌더 **자동 중단**(렌더 무거움) → 오래 안 걸림.
- ⚠️ **LIBERO 시뮬 필요**. 별도 폴더(`videos_libero/`)라 실제 eval 결과 안 건드림.


In [ ]:
import sys, json, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK = 'libero_10'
# ── 영상 뽑을 모델/seed ──  ours = bimamba_s7. 학습된 seed 로.
MODEL = 'bimamba_s7'
SEED  = 2                    # 학습·유효한 seed (상태 보고 바꿔도 됨)
N_EP  = 20                   # task당 굴릴 에피소드(성공 1개 나올 만큼). 성공 나오면 렌더 자동 중단.
PER_TASK = 1                 # task당 저장할 성공 영상 수
N_TASKS = 10

# 비표준 태그 등록(bimamba_s7=ours=bimamba_mosaic 정책)
cf.v23.MODEL_DIR_NAMES.setdefault(MODEL, MODEL)
cf.v23.MODEL_CONFIGS.setdefault(MODEL, cf.v23.MODEL_CONFIGS['bimamba_mosaic'])

cd = cf.v23.best_ckpt_dir(MODEL, SEED, TASK, how=cf.CKPT_STEP)   # 150k 근처
print('모델:', MODEL, 'seed', SEED, '| task당', PER_TASK, '성공 영상 | n_ep', N_EP)
if cd is None:
    print('⚠️ 체크포인트 없음 — SEED 바꾸거나 먼저 학습. (train/libero_10/%s/seed%d)' % (MODEL, SEED))
else:
    print('체크포인트:', cd)

## 녹화 실행


In [ ]:
# ── 성공 영상 녹화 실행 ── RECORD_VIDEOS=PER_TASK + RECORD_SUCCESS_ONLY=1 → task당 성공만 저장 ──
#    별도 폴더(videos_libero/…)에 저장 → 실제 eval 결과(eval_clean) 안 건드림. LIBERO 시뮬 필요.
ckpt = cf.v23._pretrained(cd)
OUT = cf.OUTPUT_BASE / 'videos_libero' / MODEL / f'seed{SEED}'
OUT.mkdir(parents=True, exist_ok=True)
gpu = cf.available_gpus()[0]

parts = [f'RECORD_VIDEOS={PER_TASK}', 'RECORD_SUCCESS_ONLY=1'] + cf.v23._gpu_env(gpu) + [
    f'{cf.v23.PYTHON} -m lerobot.scripts.lerobot_eval', f'--policy.path={ckpt}',
    '--env.type=libero', f'--env.task={TASK}',
    f'--eval.n_episodes={N_EP}', f'--eval.batch_size={cf.LIBERO_EVAL_BATCH}',
    f'--output_dir={OUT}',
]
cmd = ' '.join(parts)
print('GPU', gpu, '| 저장 위치:', OUT / 'videos')
print('성공 1개 나오면 그 task 렌더 중단 → 오래 안 걸림. (LIBERO 시뮬 필요)')
cf.v23.launch_cmds_live([(f'videos/{MODEL}/seed{SEED}', cmd)], log_tag='videos')

## 10 task 성공 영상 모으기 + zip


In [ ]:
# ── 10 task 성공 영상 모으기 → success_videos/task_00.mp4 … + zip ──
vroot = OUT / 'videos'                      # run_one 이 task 별로 libero_10_<id>/ 에 저장
dst = OUT / 'success_videos'
dst.mkdir(parents=True, exist_ok=True)
got, missing = [], []
for i in range(N_TASKS):
    tdir = vroot / f'{TASK}_{i}'
    succ = sorted(tdir.glob('*_success.mp4')) if tdir.is_dir() else []
    if succ:
        shutil.copy(succ[0], dst / f'task_{i:02d}.mp4')
        got.append(i)
    else:
        missing.append(i)
print(f'성공 영상 확보: {len(got)}/{N_TASKS} task  → {dst}')
if missing:
    print(f'⚠️ 성공 영상 없는 task: {missing}  (그 task 는 {N_EP}ep 안에 성공 0 → N_EP 늘려 재실행)')
else:
    print('✅ 10 task 전부 성공 영상 확보')
if got:
    zip_path = shutil.make_archive(str(OUT / f'{MODEL}_seed{SEED}_success_videos'), 'zip', root_dir=dst)
    print('보낼 파일:', zip_path)
    for p in sorted(dst.glob('*.mp4')):
        print(f'  {p.name}  ({p.stat().st_size/1024:.0f} KB)')